# HW08-09 — PyTorch MLP, регуляризация и оптимизация

В этом ноутбуке реализуем полный пайплайн, требуемый в задании:

- загрузка датасета из `torchvision` (по умолчанию `KMNIST`), разбиение на train/val/test;
- MLP-модель в `nn.Module` с возможностью включать Dropout/BatchNorm;
- корректный train/eval-цикл с логированием метрик по эпохам;
- эксперименты **E1-E4** (регуляризация, EarlyStopping) и **O1-O3** (LR, Adam vs SGD+momentum + weight decay);
- сохранение артефактов в `homeworks/HW08-09/artifacts/`: `runs.csv`, `best_model.pt`, `best_config.json`, графики кривых обучения.

In [1]:
import sys

# ставим/обновляем pip в этом же интерпретаторе
!"{sys.executable}" -m pip install --upgrade pip

# ставим PyTorch + torchvision (CPU-версия)
!"{sys.executable}" -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu


In [2]:
import os
from pathlib import Path
import json
import math
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms


# Фиксируем seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Определяем устройство
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [3]:
# Настройка путей и директорий для артефактов

# Пытаемся аккуратно определить папку HW08-09 независимо от рабочей директории
cwd = Path().resolve()

if (cwd / "artifacts").exists() and (cwd / "HW08-09.ipynb").exists():
    # Ноутбук запущен из папки homeworks/HW08-09
    HW_DIR = cwd
elif (cwd / "homeworks" / "HW08-09").exists():
    # Ноутбук запущен из корня репозитория
    HW_DIR = cwd / "homeworks" / "HW08-09"
else:
    # Фоллбек — считаем текущую папку папкой HW08-09
    HW_DIR = cwd

ARTIFACTS_DIR = HW_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
RUNS_CSV_PATH = ARTIFACTS_DIR / "runs.csv"
BEST_MODEL_PATH = ARTIFACTS_DIR / "best_model.pt"
BEST_CONFIG_PATH = ARTIFACTS_DIR / "best_config.json"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

HW_DIR, ARTIFACTS_DIR, FIGURES_DIR

(WindowsPath('C:/Users/Гошанский/PycharmProjects/mirea-aie/homeworks/HW08-09'),
 WindowsPath('C:/Users/Гошанский/PycharmProjects/mirea-aie/homeworks/HW08-09/artifacts'),
 WindowsPath('C:/Users/Гошанский/PycharmProjects/mirea-aie/homeworks/HW08-09/artifacts/figures'))

In [4]:
# Данные и DataLoader

DATASET_NAME = "EMNIST-balanced"  # варианты из условия: KMNIST / EMNIST-balanced / CIFAR10
BATCH_SIZE = 128
VAL_FRACTION = 0.2

if DATASET_NAME == "KMNIST":
    num_classes = 10
    input_channels = 1
    image_size = 28
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_full = torchvision.datasets.KMNIST(
        root=str(HW_DIR / "data"),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = torchvision.datasets.KMNIST(
        root=str(HW_DIR / "data"),
        train=False,
        download=True,
        transform=transform,
    )

elif DATASET_NAME == "EMNIST-balanced":
    num_classes = 47
    input_channels = 1
    image_size = 28
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_full = torchvision.datasets.EMNIST(
        root=str(HW_DIR / "data"),
        split="balanced",
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = torchvision.datasets.EMNIST(
        root=str(HW_DIR / "data"),
        split="balanced",
        train=False,
        download=True,
        transform=transform,
    )

elif DATASET_NAME == "CIFAR10":
    num_classes = 10
    input_channels = 3
    image_size = 32
    transform = transforms.Compose([        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    train_full = torchvision.datasets.CIFAR10(
        root=str(HW_DIR / "data"),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = torchvision.datasets.CIFAR10(
        root=str(HW_DIR / "data"),
        train=False,
        download=True,
        transform=transform,
    )

else:
    raise ValueError(f"Unknown DATASET_NAME={DATASET_NAME}")

# Разбиение train -> train/val с фиксированным seed
num_train = len(train_full)
num_val = int(num_train * VAL_FRACTION)
num_train_actual = num_train - num_val

train_dataset, val_dataset = random_split(
    train_full,
    [num_train_actual, num_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# sanity-check батча
x_batch, y_batch = next(iter(train_loader))
print("Batch shapes:", x_batch.shape, y_batch.shape)
print("x min/max:", float(x_batch.min()), float(x_batch.max()))
print("num_train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))

Batch shapes: torch.Size([128, 1, 28, 28]) torch.Size([128])
x min/max: -1.0 1.0
num_train/val/test: 90240 22560 18800


In [5]:
# Определение MLP-модели с опциями Dropout и BatchNorm

class MLP(nn.Module):
    def __init__(
        self,
        input_channels: int,
        image_size: int,
        num_classes: int,
        hidden_sizes=(256, 256, 128),
        activation=nn.ReLU,
        use_dropout: bool = False,
        dropout_p: float = 0.3,
        use_batchnorm: bool = False,
    ):
        super().__init__()
        self.input_dim = input_channels * image_size * image_size

        layers = [nn.Flatten()]
        in_dim = self.input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(activation())
            if use_dropout:
                layers.append(nn.Dropout(p=dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))  # logits

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Пример и проверка формы выхода
model_example = MLP(input_channels=input_channels, image_size=image_size, num_classes=num_classes)
print(model_example)
print("Trainable params:", count_parameters(model_example))

x_batch, _ = next(iter(train_loader))
with torch.no_grad():
    logits = model_example(x_batch)
print("Logits shape:", logits.shape)

MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): ReLU()
    (5): Linear(in_features=256, out_features=128, bias=True)
    (6): ReLU()
    (7): Linear(in_features=128, out_features=47, bias=True)
  )
)
Trainable params: 305711
Logits shape: torch.Size([128, 47])


In [6]:
# Вспомогательные функции: accuracy, train_one_epoch, evaluate

criterion = nn.CrossEntropyLoss()


def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    correct = (preds == targets).sum().item()
    return correct / targets.size(0)


def train_one_epoch(model, dataloader, optimizer, device=DEVICE):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, device=DEVICE):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            batch_size = y.size(0)
            running_loss += loss.item() * batch_size
            running_correct += (logits.argmax(dim=1) == y).sum().item()
            running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc

In [7]:
# Примитивный EarlyStopping по val_loss или val_accuracy

class EarlyStopping:
    def __init__(self, mode: str = "min", patience: int = 5):
        assert mode in {"min", "max"}
        self.mode = mode
        self.patience = patience
        self.best_score = None
        self.best_state_dict = None
        self.num_bad_epochs = 0

    def step(self, score: float, model: nn.Module):
        if self.best_score is None:
            self.best_score = score
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.num_bad_epochs = 0
            return False  # не останавливаемся

        improvement = (score < self.best_score) if self.mode == "min" else (score > self.best_score)
        if improvement:
            self.best_score = score
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.num_bad_epochs = 0
            return False
        else:
            self.num_bad_epochs += 1
            return self.num_bad_epochs > self.patience

    def restore_best_weights(self, model: nn.Module):
        if self.best_state_dict is not None:
            model.load_state_dict(self.best_state_dict)


# Проверка, что класс создается
_ = EarlyStopping(mode="max", patience=3)

In [8]:
# Обвязка для экспериментов: единая функция запуска и логирования в runs.csv

import csv


all_runs = []  # список словарей с результатами по всем экспериментам
histories = {}  # experiment_id -> history dict


def model_summary_str(hidden_sizes, activation_name, use_dropout, dropout_p, use_batchnorm):
    return (
        f"MLP(hidden={list(hidden_sizes)}, act={activation_name}, "
        f"dropout={dropout_p if use_dropout else 0}, bn={use_batchnorm})"
    )


def append_run_to_csv(run_row: dict, csv_path: Path = RUNS_CSV_PATH):
    fieldnames = [
        "experiment_id",
        "dataset",
        "seed",
        "model_summary",
        "optimizer",
        "lr",
        "momentum",
        "weight_decay",
        "epochs_trained",
        "best_val_accuracy",
        "best_val_loss",
    ]

    file_exists = csv_path.exists()
    with csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(run_row)


def run_experiment(
    experiment_id: str,
    hidden_sizes=(256, 256, 128),
    use_dropout: bool = False,
    dropout_p: float = 0.3,
    use_batchnorm: bool = False,
    optimizer_name: str = "Adam",
    lr: float = 1e-3,
    momentum: float = 0.0,
    weight_decay: float = 0.0,
    max_epochs: int = 15,
    use_early_stopping: bool = False,
    early_stopping_patience: int = 5,
    early_stopping_mode: str = "max",  # по умолчанию по val_accuracy
    device=DEVICE,
):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    model = MLP(
        input_channels=input_channels,
        image_size=image_size,
        num_classes=num_classes,
        hidden_sizes=hidden_sizes,
        use_dropout=use_dropout,
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
    ).to(device)

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay
        )
    else:
        raise ValueError(f"Unknown optimizer_name={optimizer_name}")

    es = EarlyStopping(mode=early_stopping_mode, patience=early_stopping_patience) if use_early_stopping else None

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = -math.inf
    best_val_loss = math.inf
    epochs_trained = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device=device)
        val_loss, val_acc = evaluate(model, val_loader, device=device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        epochs_trained = epoch

        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        print(
            f"[{experiment_id}] Epoch {epoch}/{max_epochs} "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}"
        )

        if es is not None:
            # следим за метрикой в зависимости от режима (по условию best по val_accuracy)
            score = val_acc if early_stopping_mode == "max" else val_loss
            should_stop = es.step(score, model)
            if should_stop:
                print(f"EarlyStopping: stop at epoch {epoch}")
                break

    if es is not None:
        es.restore_best_weights(model)

    histories[experiment_id] = history

    run_row = {
        "experiment_id": experiment_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": model_summary_str(
            hidden_sizes, activation_name="ReLU", use_dropout=use_dropout,
            dropout_p=dropout_p, use_batchnorm=use_batchnorm,
        ),
        "optimizer": optimizer_name,
        "lr": lr,
        "momentum": momentum,
        "weight_decay": weight_decay,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
    }

    all_runs.append(run_row)
    append_run_to_csv(run_row)

    return model, history, run_row

In [9]:
# Эксперименты E1–E3: базовая модель, Dropout, BatchNorm

BASE_HIDDEN_SIZES = (256, 256, 128)
BASE_LR = 1e-3
BASE_EPOCHS = 12  # чуть укорочено, чтобы не было слишком долго на CPU

# E1: base — без Dropout и BatchNorm
model_E1, history_E1, run_E1 = run_experiment(
    experiment_id="E1",
    hidden_sizes=BASE_HIDDEN_SIZES,
    use_dropout=False,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    use_early_stopping=False,
)

# E2: как E1, но с Dropout
model_E2, history_E2, run_E2 = run_experiment(
    experiment_id="E2",
    hidden_sizes=BASE_HIDDEN_SIZES,
    use_dropout=True,
    dropout_p=0.3,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    use_early_stopping=False,
)

# E3: как E1, но с BatchNorm
model_E3, history_E3, run_E3 = run_experiment(
    experiment_id="E3",
    hidden_sizes=BASE_HIDDEN_SIZES,
    use_dropout=False,
    use_batchnorm=True,
    optimizer_name="Adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    use_early_stopping=False,
)

run_E1, run_E2, run_E3

[E1] Epoch 1/12 train_loss=1.2681 val_loss=0.8232 train_acc=0.6283 val_acc=0.7406
[E1] Epoch 2/12 train_loss=0.6832 val_loss=0.6298 train_acc=0.7798 val_acc=0.7972
[E1] Epoch 3/12 train_loss=0.5666 val_loss=0.5741 train_acc=0.8101 val_acc=0.8157
[E1] Epoch 4/12 train_loss=0.5045 val_loss=0.5326 train_acc=0.8278 val_acc=0.8245
[E1] Epoch 5/12 train_loss=0.4640 val_loss=0.5168 train_acc=0.8399 val_acc=0.8275
[E1] Epoch 6/12 train_loss=0.4311 val_loss=0.5072 train_acc=0.8476 val_acc=0.8318
[E1] Epoch 7/12 train_loss=0.4046 val_loss=0.4881 train_acc=0.8554 val_acc=0.8393
[E1] Epoch 8/12 train_loss=0.3868 val_loss=0.5034 train_acc=0.8590 val_acc=0.8329
[E1] Epoch 9/12 train_loss=0.3649 val_loss=0.5132 train_acc=0.8665 val_acc=0.8295
[E1] Epoch 10/12 train_loss=0.3507 val_loss=0.5024 train_acc=0.8697 val_acc=0.8323
[E1] Epoch 11/12 train_loss=0.3321 val_loss=0.4953 train_acc=0.8750 val_acc=0.8389
[E1] Epoch 12/12 train_loss=0.3193 val_loss=0.4895 train_acc=0.8784 val_acc=0.8404
[E2] Epoch 1/

({'experiment_id': 'E1',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=False)',
  'optimizer': 'Adam',
  'lr': 0.001,
  'momentum': 0.0,
  'weight_decay': 0.0,
  'epochs_trained': 12,
  'best_val_accuracy': 0.8403812056737588,
  'best_val_loss': 0.488138554535859},
 {'experiment_id': 'E2',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0.3, bn=False)',
  'optimizer': 'Adam',
  'lr': 0.001,
  'momentum': 0.0,
  'weight_decay': 0.0,
  'epochs_trained': 12,
  'best_val_accuracy': 0.8195035460992908,
  'best_val_loss': 0.534700338468484},
 {'experiment_id': 'E3',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=True)',
  'optimizer': 'Adam',
  'lr': 0.001,
  'momentum': 0.0,
  'weight_decay': 0.0,
  'epochs_trained': 12,
  'best_val_accuracy': 0.8521719858156028,
  'best_val_loss': 0.440

In [10]:
# Эксперимент E4: EarlyStopping для лучшей из (E2, E3) по val_accuracy

# Выбираем, что лучше по val_accuracy
val_acc_E2 = run_E2["best_val_accuracy"]
val_acc_E3 = run_E3["best_val_accuracy"]

if val_acc_E2 >= val_acc_E3:
    best_reg_experiment = "E2"
    use_dropout = True
    dropout_p = 0.3
    use_batchnorm = False
else:
    best_reg_experiment = "E3"
    use_dropout = False
    dropout_p = 0.0
    use_batchnorm = True

print(f"Лучший по val_accuracy среди (E2, E3): {best_reg_experiment}")

# E4: тот же конфиг, но с EarlyStopping (patience 4) — лучшая модель домашки
model_E4, history_E4, run_E4 = run_experiment(
    experiment_id="E4",
    hidden_sizes=BASE_HIDDEN_SIZES,
    use_dropout=use_dropout,
    dropout_p=dropout_p,
    use_batchnorm=use_batchnorm,
    optimizer_name="Adam",
    lr=BASE_LR,
    max_epochs=30,
    use_early_stopping=True,
    early_stopping_patience=4,
    early_stopping_mode="max",
)

run_E4

Лучший по val_accuracy среди (E2, E3): E3
[E4] Epoch 1/30 train_loss=0.9366 val_loss=0.5684 train_acc=0.7385 val_acc=0.8137
[E4] Epoch 2/30 train_loss=0.5045 val_loss=0.5112 train_acc=0.8295 val_acc=0.8298
[E4] Epoch 3/30 train_loss=0.4310 val_loss=0.4626 train_acc=0.8492 val_acc=0.8420
[E4] Epoch 4/30 train_loss=0.3843 val_loss=0.4582 train_acc=0.8617 val_acc=0.8437
[E4] Epoch 5/30 train_loss=0.3502 val_loss=0.4409 train_acc=0.8729 val_acc=0.8508
[E4] Epoch 6/30 train_loss=0.3239 val_loss=0.4488 train_acc=0.8792 val_acc=0.8477
[E4] Epoch 7/30 train_loss=0.3028 val_loss=0.4441 train_acc=0.8847 val_acc=0.8522
[E4] Epoch 8/30 train_loss=0.2819 val_loss=0.4529 train_acc=0.8918 val_acc=0.8480
[E4] Epoch 9/30 train_loss=0.2671 val_loss=0.4461 train_acc=0.8964 val_acc=0.8520
[E4] Epoch 10/30 train_loss=0.2511 val_loss=0.4556 train_acc=0.9019 val_acc=0.8518
[E4] Epoch 11/30 train_loss=0.2389 val_loss=0.4506 train_acc=0.9054 val_acc=0.8504
[E4] Epoch 12/30 train_loss=0.2271 val_loss=0.4739 tra

{'experiment_id': 'E4',
 'dataset': 'EMNIST-balanced',
 'seed': 42,
 'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=True)',
 'optimizer': 'Adam',
 'lr': 0.001,
 'momentum': 0.0,
 'weight_decay': 0.0,
 'epochs_trained': 12,
 'best_val_accuracy': 0.8521719858156028,
 'best_val_loss': 0.44093856105567714}

In [11]:
# Часть B (S09): эксперименты O1–O3 (diagnostics по LR и SGD+momentum + weight decay)

LR_TOO_BIG = 1e-1
LR_TOO_SMALL = 1e-5
SGD_LR = 3e-3  # разумный LR под SGD
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# Фиксируем архитектуру и регуляризацию как в E4
O_HIDDEN_SIZES = BASE_HIDDEN_SIZES
O_USE_DROPOUT = use_dropout
O_DROPOUT_P = dropout_p
O_USE_BATCHNORM = use_batchnorm

# O1: Adam, слишком большой LR
model_O1, history_O1, run_O1 = run_experiment(
    experiment_id="O1",
    hidden_sizes=O_HIDDEN_SIZES,
    use_dropout=O_USE_DROPOUT,
    dropout_p=O_DROPOUT_P,
    use_batchnorm=O_USE_BATCHNORM,
    optimizer_name="Adam",
    lr=LR_TOO_BIG,
    max_epochs=8,
    use_early_stopping=False,
)

# O2: Adam, слишком маленький LR
model_O2, history_O2, run_O2 = run_experiment(
    experiment_id="O2",
    hidden_sizes=O_HIDDEN_SIZES,
    use_dropout=O_USE_DROPOUT,
    dropout_p=O_DROPOUT_P,
    use_batchnorm=O_USE_BATCHNORM,
    optimizer_name="Adam",
    lr=LR_TOO_SMALL,
    max_epochs=8,
    use_early_stopping=False,
)

# O3: SGD + momentum + weight decay
model_O3, history_O3, run_O3 = run_experiment(
    experiment_id="O3",
    hidden_sizes=O_HIDDEN_SIZES,
    use_dropout=O_USE_DROPOUT,
    dropout_p=O_DROPOUT_P,
    use_batchnorm=O_USE_BATCHNORM,
    optimizer_name="SGD",
    lr=SGD_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    max_epochs=12,
    use_early_stopping=False,
)

run_O1, run_O2, run_O3

[O1] Epoch 1/8 train_loss=1.1057 val_loss=0.8182 train_acc=0.6642 val_acc=0.7352
[O1] Epoch 2/8 train_loss=0.7442 val_loss=0.7018 train_acc=0.7599 val_acc=0.7766
[O1] Epoch 3/8 train_loss=0.6673 val_loss=0.6148 train_acc=0.7819 val_acc=0.8007
[O1] Epoch 4/8 train_loss=0.6163 val_loss=0.6393 train_acc=0.7942 val_acc=0.7977
[O1] Epoch 5/8 train_loss=0.5878 val_loss=0.6143 train_acc=0.8032 val_acc=0.8025
[O1] Epoch 6/8 train_loss=0.5604 val_loss=0.5858 train_acc=0.8118 val_acc=0.8160
[O1] Epoch 7/8 train_loss=0.5429 val_loss=0.5891 train_acc=0.8172 val_acc=0.8118
[O1] Epoch 8/8 train_loss=0.5200 val_loss=0.5788 train_acc=0.8226 val_acc=0.8165
[O2] Epoch 1/8 train_loss=3.4911 val_loss=3.1298 train_acc=0.1872 val_acc=0.3611
[O2] Epoch 2/8 train_loss=2.9060 val_loss=2.6961 train_acc=0.4379 val_acc=0.4978
[O2] Epoch 3/8 train_loss=2.5405 val_loss=2.3789 train_acc=0.5344 val_acc=0.5698
[O2] Epoch 4/8 train_loss=2.2571 val_loss=2.1168 train_acc=0.5898 val_acc=0.6139
[O2] Epoch 5/8 train_loss=2.

({'experiment_id': 'O1',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=True)',
  'optimizer': 'Adam',
  'lr': 0.1,
  'momentum': 0.0,
  'weight_decay': 0.0,
  'epochs_trained': 8,
  'best_val_accuracy': 0.8165336879432624,
  'best_val_loss': 0.5787844193742631},
 {'experiment_id': 'O2',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=True)',
  'optimizer': 'Adam',
  'lr': 1e-05,
  'momentum': 0.0,
  'weight_decay': 0.0,
  'epochs_trained': 8,
  'best_val_accuracy': 0.7052748226950355,
  'best_val_loss': 1.4422051417912152},
 {'experiment_id': 'O3',
  'dataset': 'EMNIST-balanced',
  'seed': 42,
  'model_summary': 'MLP(hidden=[256, 256, 128], act=ReLU, dropout=0, bn=True)',
  'optimizer': 'SGD',
  'lr': 0.003,
  'momentum': 0.9,
  'weight_decay': 0.0001,
  'epochs_trained': 12,
  'best_val_accuracy': 0.8478280141843971,
  'best_val_loss': 0.4557415

In [12]:
# Графики кривых обучения: лучший прогон (E4) и экстремальные LR (O1, O2)
# Используем Pillow вместо matplotlib, чтобы избежать падений ядра.

from PIL import Image, ImageDraw


def _draw_curves_png(path, series_dict, title, width=800, height=400, margin=50):
    """Примитивный рисователь нескольких кривых (epoch -> значение) в PNG.

    series_dict: {name: list_of_values}
    """
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)

    # Подготовка данных
    max_len = max(len(vals) for vals in series_dict.values())
    xs = list(range(max_len))
    all_vals = [v for vals in series_dict.values() for v in vals]
    v_min, v_max = min(all_vals), max(all_vals)
    if v_max == v_min:
        v_max = v_min + 1e-6

    def to_xy(idx, v):
        x = margin + (width - 2 * margin) * (idx / max(1, max_len - 1))
        # по Y инвертируем, чтобы большие значения были выше
        y = height - margin - (height - 2 * margin) * ((v - v_min) / (v_max - v_min))
        return x, y

    # Оси
    draw.line((margin, margin, margin, height - margin), fill="black", width=1)
    draw.line((margin, height - margin, width - margin, height - margin), fill="black", width=1)

    colors = ["red", "blue", "green", "orange", "purple"]

    for (name, vals), color in zip(series_dict.items(), colors):
        if len(vals) < 2:
            continue
        points = [to_xy(i, v) for i, v in enumerate(vals)]
        draw.line(points, fill=color, width=2)
        # подпись в правом верхнем углу (просто сетка по вертикали)
        idx = list(series_dict.keys()).index(name)
        legend_y = margin + idx * 15
        draw.text((margin + 5, legend_y), name, fill=color)

    # Заголовок
    draw.text((margin, 5), title, fill="black")

    img.save(path)
    print("Saved", path)


# E4: train/val loss + accuracy
h = histories["E4"]
curves_best_path = FIGURES_DIR / "curves_best.png"
_draw_curves_png(
    curves_best_path,
    {
        "train_loss": h["train_loss"],
        "val_loss": h["val_loss"],
        "train_acc": h["train_acc"],
        "val_acc": h["val_acc"],
    },
    title="E4: loss + acc",
)

# O1/O2: только loss, чтобы показать плохой LR
h_O1 = histories["O1"]
h_O2 = histories["O2"]
curves_lr_extremes_path = FIGURES_DIR / "curves_lr_extremes.png"
_draw_curves_png(
    curves_lr_extremes_path,
    {
        f"O1 train_loss (lr={LR_TOO_BIG})": h_O1["train_loss"],
        f"O1 val_loss (lr={LR_TOO_BIG})": h_O1["val_loss"],
        f"O2 train_loss (lr={LR_TOO_SMALL})": h_O2["train_loss"],
        f"O2 val_loss (lr={LR_TOO_SMALL})": h_O2["val_loss"],
    },
    title="O1/O2: bad LR",
)

Saved C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW08-09\artifacts\figures\curves_best.png
Saved C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW08-09\artifacts\figures\curves_lr_extremes.png


In [13]:
# Финальная оценка лучшей модели (E4) на test и сохранение best_model.pt и best_config.json

# Убеждаемся, что модель E4 на нужном устройстве
model_E4 = model_E4.to(DEVICE)

# Оценка на test (один раз, после выбора по val)
test_loss, test_acc = evaluate(model_E4, test_loader, device=DEVICE)
print(f"[E4 on TEST] loss={test_loss:.4f}, acc={test_acc:.4f}")

# Сохранение state_dict лучшей модели
torch.save(model_E4.state_dict(), BEST_MODEL_PATH)
print("Saved best_model.pt ->", BEST_MODEL_PATH)

# Сохранение best_config.json
best_config = {
    "experiment_id": "E4",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "hidden_sizes": list(BASE_HIDDEN_SIZES),
    "use_dropout": bool(O_USE_DROPOUT),
    "dropout_p": float(O_DROPOUT_P),
    "use_batchnorm": bool(O_USE_BATCHNORM),
    "optimizer": "Adam",
    "lr": float(BASE_LR),
    "weight_decay": 0.0,
    "early_stopping": True,
    "early_stopping_patience": 4,
}

with BEST_CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2, ensure_ascii=False)

BEST_MODEL_PATH, BEST_CONFIG_PATH

[E4 on TEST] loss=0.4608, acc=0.8468
Saved best_model.pt -> C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW08-09\artifacts\best_model.pt


(WindowsPath('C:/Users/Гошанский/PycharmProjects/mirea-aie/homeworks/HW08-09/artifacts/best_model.pt'),
 WindowsPath('C:/Users/Гошанский/PycharmProjects/mirea-aie/homeworks/HW08-09/artifacts/best_config.json'))